### Data Quality Check
---
- This notebook is focused upon the quality of data before moving to processing and prediction
- The goal is to observe whether the data has inconsistencies, missing value and other noise or not.

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/medical_appointment_no_show_raw.csv')

In [2]:
df.isnull().sum()

PatientId         0
AppointmentID     0
Gender            0
ScheduledDay      0
AppointmentDay    0
Age               0
Neighbourhood     0
Scholarship       0
Hipertension      0
Diabetes          0
Alcoholism        0
Handcap           0
SMS_received      0
No-show           0
dtype: int64

### Key Observation
---
 The dataset has no missing values which shows data completeness and is very useful in further processing and accurate prediction.

In [3]:
df['Age'].describe()


count    110527.000000
mean         37.088874
std          23.110205
min          -1.000000
25%          18.000000
50%          37.000000
75%          55.000000
max         115.000000
Name: Age, dtype: float64

In [4]:
df[df['Age']<0]


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
99832,4.659432e+14,5775010,F,2016-06-06T08:58:13Z,2016-06-06T00:00:00Z,-1,ROMÃO,0,0,0,0,0,0,No


In [5]:
df[df['Age']>100]

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
58014,9.762948e+14,5651757,F,2016-05-03T09:14:53Z,2016-05-03T00:00:00Z,102,CONQUISTA,0,0,0,0,0,0,No
63912,3.196321e+13,5700278,F,2016-05-16T09:17:44Z,2016-05-19T00:00:00Z,115,ANDORINHAS,0,0,0,0,1,0,Yes
63915,3.196321e+13,5700279,F,2016-05-16T09:17:44Z,2016-05-19T00:00:00Z,115,ANDORINHAS,0,0,0,0,1,0,Yes
68127,3.196321e+13,5562812,F,2016-04-08T14:29:17Z,2016-05-16T00:00:00Z,115,ANDORINHAS,0,0,0,0,1,0,Yes
76284,3.196321e+13,5744037,F,2016-05-30T09:44:51Z,2016-05-30T00:00:00Z,115,ANDORINHAS,0,0,0,0,1,0,No
90372,2.342836e+11,5751563,F,2016-05-31T10:19:49Z,2016-06-02T00:00:00Z,102,MARIA ORTIZ,0,0,0,0,0,0,No
97666,7.482346e+14,5717451,F,2016-05-19T07:57:56Z,2016-06-03T00:00:00Z,115,SÃO JOSÉ,0,1,0,0,0,1,No


### Invalid or Suspicious Values
---
**Age**
- In the dataset there is one patient who's age is less than 0 which is not possible
- On the other hand there are some patients with the age more than 100 which is quite abnormal which will require validation.

**Decision from Observation**
- Based on the observation the negative age is impossible biologically so it is required to be fixed.
- The ages above 100 is not impossible and removing them can lead to biasness so we will retain them.

In [6]:
df = df[df['Age'] > 0]
df['Age'].min()

np.int64(1)

The record where age was less than 0 is successfully dropped.

In [7]:
df['Handcap'].value_counts()

Handcap
0    104747
1      2041
2       183
3        13
4         3
Name: count, dtype: int64

In [8]:
df['Handcap_binary'] = (df['Handcap']>0).astype(int)
df['Handcap_binary'].value_counts()

Handcap_binary
0    104747
1      2240
Name: count, dtype: int64

### Handcap Observation

- It represents if a patient has disabilities or not but is was not a binary coloumn
- Values more than 1 could have added noise
- The values of this feature is tranformed into 0 or 1 form in order to avoid problem while model training where :

    - 0 shows that patient do have any disability
    - 1 shows that patient is disabled


In [9]:
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

In [10]:
(df['AppointmentDay'] < df['ScheduledDay']).sum()


np.int64(37159)

In [11]:
df = df[df['AppointmentDay'] >= df['ScheduledDay']]

### Date Consistency Check
---
- It is verified that no appointment day come before the scheduled day.
- If any then it is removed as it will bring illogical error in the further processing.

In [12]:
df['WaitingDays'] = (df['AppointmentDay'] - df['ScheduledDay']).dt.days
df['WaitingDays'].describe()

count    69828.000000
mean        14.577089
std         16.512813
min          0.000000
25%          3.000000
50%          8.000000
75%         21.000000
max        178.000000
Name: WaitingDays, dtype: float64

### Waiting Days Feature
---
- This feature is added in order to calculate how many days patient has to wait for appointment.
- This is directly related to the patient behavior as he shows on appointment day or not as longer waiting time mostly resulting in absence on the appointed day.

In [13]:
df['No-show'].value_counts(normalize=True)*100

No-show
No     71.452712
Yes    28.547288
Name: proportion, dtype: float64

In [14]:
df[['Scholarship','Hipertension','Diabetes','Alcoholism','Handcap_binary','SMS_received']].nunique()

Scholarship       2
Hipertension      2
Diabetes          2
Alcoholism        2
Handcap_binary    2
SMS_received      2
dtype: int64

In [15]:
df.to_csv('../data/processed/medical_appointment_no_show_cleaned.csv', index=False)


### Data Quality Checks Summary
---

- Invalid values in `Age` column is fixed so that there are no negative values in this feature.
- A new `Handcap_binary` column is derived from the feature so `handcap` to bring uniformity so that number of disabilities is represnted by 1 only also preserving the original feature.
- Date Consistency is applied in order to confirm that appointment day is never before scheduled day and waiting feature is applied which can be helpful in No-show prediction.
- Verification of target variable and binary features as they contain type of values as expected.
